# 02 — Image Feature Extraction for v1.0-trainval Subset
## Multi-Attribute Scene Classification on nuScenes

**Purpose:** Extract hand-crafted classical features from the ~6,021 CAM_FRONT images covering our 150 selected scenes.

### Feature groups (identical to Stage 1)

| Group | Description | Dimensions |
|---|---|---|
| **HOG** | Histogram of Oriented Gradients | ~3,000 |
| **Color** | RGB + HSV histograms | 96 |
| **LBP** | Local Binary Patterns | 26 |
| **Photometric** | Brightness, contrast, saturation statistics | 10 |
| **Total** | | ~6,200 |

### Parameters (identical to Stage 1 — notebook 03)

- **Image size**: 224 × 224 (downscaled for HOG/LBP)
- **HOG**: 9 orientations, 8×8 pixels per cell, 2×2 cells per block
- **Color**: 32 bins per channel for both RGB and HSV
- **LBP**: radius=3, n_points=24, uniform method, 26 bins
- **Photometric**: 10 statistical features (mean, std, etc. across channels)

### Parallelization

Uses `joblib` to parallelize across CPU cores. On a 16-core machine, expect ~10-15 min total.

### Inputs
- `data/processed/v1.0-trainval/labels/attribute_labels.csv` (from 01)
- `data/nuscenes/v1.0-trainval/samples/CAM_FRONT/*.jpg` (from 00c)

### Outputs
- `data/processed/v1.0-trainval/features/features_full.csv`
- `data/processed/v1.0-trainval/features/feature_metadata.json`


In [1]:
# %pip install scikit-image joblib tqdm

## 0. Setup

In [2]:
import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

from skimage.feature import hog, local_binary_pattern
from skimage.color import rgb2hsv
from skimage import img_as_float
from joblib import Parallel, delayed
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

print('Imports OK')

Imports OK


## 0.1 Locate Project Root

In [3]:
def find_project_root():
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError(f'Could not find project root from {p}')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\leemi\Documents\GitHub\nuscenes-scene-classification-ml


## 1. Configuration

In [4]:
DATASET_VERSION = 'v1.0-trainval'

# Paths
NUSCENES_BASE = Path('data/nuscenes') / DATASET_VERSION
IMG_DIR       = NUSCENES_BASE / 'samples' / 'CAM_FRONT'
PROCESSED_DIR = Path('data/processed') / DATASET_VERSION
LABELS_PATH   = PROCESSED_DIR / 'labels' / 'attribute_labels.csv'
FEATURES_DIR  = PROCESSED_DIR / 'features'
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# ─── Feature parameters (MUST match Stage 1 exactly) ─────────────────
IMG_SIZE = 224  # downscale for HOG/LBP

HOG_PARAMS = dict(
    orientations=9,
    pixels_per_cell=(16, 16),
    cells_per_block=(2, 2),
    block_norm='L2-Hys',
    transform_sqrt=True,
    feature_vector=True,
)

COLOR_BINS = 32   # per channel, for both RGB and HSV
LBP_RADIUS = 3
LBP_N_POINTS = 24
LBP_METHOD = 'uniform'
LBP_BINS = LBP_N_POINTS + 2

# Parallelization
N_JOBS = -1  # use all cores

print(f'DATASET_VERSION = {DATASET_VERSION}')
print(f'IMG_DIR         = {IMG_DIR}')
print(f'LABELS_PATH     = {LABELS_PATH}')
print(f'IMG_SIZE        = {IMG_SIZE}')
print(f'N_JOBS          = {N_JOBS} (all cores)')

DATASET_VERSION = v1.0-trainval
IMG_DIR         = data\nuscenes\v1.0-trainval\samples\CAM_FRONT
LABELS_PATH     = data\processed\v1.0-trainval\labels\attribute_labels.csv
IMG_SIZE        = 224
N_JOBS          = -1 (all cores)


## 2. Load Labels

In [5]:
if not LABELS_PATH.exists():
    raise FileNotFoundError(f'Run 01_attribute_labels.ipynb first to produce {LABELS_PATH}')

df_labels = pd.read_csv(LABELS_PATH)
print(f'Loaded {len(df_labels)} keyframes from {LABELS_PATH}')

# Resolve image paths
df_labels['image_path'] = df_labels['filename'].apply(
    lambda f: NUSCENES_BASE / f
)

# Verify a few images exist
sample_paths = df_labels['image_path'].sample(5, random_state=42).tolist()
print('\nSample image paths (verifying existence):')
for p in sample_paths:
    print(f'  {"✓" if p.exists() else "✗"}  {p.name}')

Loaded 6021 keyframes from data\processed\v1.0-trainval\labels\attribute_labels.csv

Sample image paths (verifying existence):
  ✓  n015-2018-11-14-19-09-14+0800__CAM_FRONT__1542194059412460.jpg
  ✓  n008-2018-08-29-16-04-13-0400__CAM_FRONT__1535573299912404.jpg
  ✓  n008-2018-09-18-13-10-39-0400__CAM_FRONT__1537290955612404.jpg
  ✓  n008-2018-08-30-15-16-55-0400__CAM_FRONT__1535656918262404.jpg
  ✓  n015-2018-11-21-19-38-26+0800__CAM_FRONT__1542800997912460.jpg


## 3. Feature Extraction Functions

In [6]:
def extract_hog(rgb_image_resized):
    """HOG features from a resized RGB image."""
    gray = np.mean(rgb_image_resized, axis=2)
    return hog(gray, **HOG_PARAMS)

def extract_color_histograms(rgb_image_full):
    """RGB color histograms only (matches Stage 1)."""
    hists = []
    for c in range(3):
        h, _ = np.histogram(rgb_image_full[..., c], bins=COLOR_BINS, range=(0, 1))
        hists.append(h)
    h_full = np.concatenate(hists).astype(np.float32)
    h_full /= (h_full.sum() + 1e-9)
    return h_full

def extract_lbp(rgb_image_resized):
    """Local Binary Patterns from grayscale."""
    gray = (np.mean(rgb_image_resized, axis=2) * 255).astype(np.uint8)
    lbp = local_binary_pattern(gray, LBP_N_POINTS, LBP_RADIUS, method=LBP_METHOD)
    h, _ = np.histogram(lbp.ravel(), bins=LBP_BINS, range=(0, LBP_BINS))
    return h / h.sum() if h.sum() > 0 else h.astype(float)

def extract_photometric(rgb_image_full):
    """10 photometric stats: brightness, contrast, saturation stats."""
    gray = np.mean(rgb_image_full, axis=2)
    hsv  = rgb2hsv(rgb_image_full)
    h_chan, s_chan, v_chan = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    feats = [
        gray.mean(),    gray.std(),    np.median(gray),
        s_chan.mean(),  s_chan.std(),
        v_chan.mean(),  v_chan.std(),
        h_chan.mean(),
        np.percentile(gray, 10),
        np.percentile(gray, 90),
    ]
    return np.array(feats, dtype=np.float32)

def extract_all_features(image_path):
    """Load an image and extract all four feature groups."""
    img = Image.open(image_path).convert('RGB')
    rgb_full = img_as_float(np.array(img))  # 0-1 floats
    img_resized = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    rgb_resized = img_as_float(np.array(img_resized))

    hog_feat   = extract_hog(rgb_resized)
    color_feat = extract_color_histograms(rgb_full)
    lbp_feat   = extract_lbp(rgb_resized)
    photo_feat = extract_photometric(rgb_full)

    return np.concatenate([hog_feat, color_feat, lbp_feat, photo_feat])

print('Feature extraction functions defined')

Feature extraction functions defined


## 4. Probe One Image to Determine Dimensions

In [7]:
probe_path = df_labels['image_path'].iloc[0]
print(f'Probing: {probe_path.name}')
t0 = time.time()
probe_feat = extract_all_features(probe_path)
t1 = time.time()

# Determine sub-dimensions
img = Image.open(probe_path).convert('RGB')
rgb_full = img_as_float(np.array(img))
img_resized = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
rgb_resized = img_as_float(np.array(img_resized))

n_hog   = len(extract_hog(rgb_resized))
n_color = len(extract_color_histograms(rgb_full))
n_lbp   = len(extract_lbp(rgb_resized))
n_photo = len(extract_photometric(rgb_full))

print(f'\n✓ Probe successful')
print(f'  HOG:         {n_hog}')
print(f'  Color:       {n_color}')
print(f'  LBP:         {n_lbp}')
print(f'  Photometric: {n_photo}')
print(f'  TOTAL:       {len(probe_feat)} (matches {n_hog + n_color + n_lbp + n_photo})')
print(f'  Time/image:  {(t1 - t0) * 1000:.0f} ms')
estimated_total_min = (t1 - t0) * len(df_labels) / 60
print(f'\nEstimated SERIAL runtime: {estimated_total_min:.1f} min')
print(f'Estimated PARALLEL runtime: {estimated_total_min / os.cpu_count() * 1.5:.1f} min '
       f'(approximate, assumes {os.cpu_count()} cores)')

Probing: n015-2018-07-18-11-50-34+0800__CAM_FRONT__1531886069512463.jpg

✓ Probe successful
  HOG:         6084
  Color:       96
  LBP:         26
  Photometric: 10
  TOTAL:       6216 (matches 6216)
  Time/image:  231 ms

Estimated SERIAL runtime: 23.2 min
Estimated PARALLEL runtime: 1.1 min (approximate, assumes 32 cores)


## 5. Parallel Feature Extraction

In [8]:
# Define a safe wrapper that captures errors
def safe_extract(path):
    try:
        return extract_all_features(path), None
    except Exception as e:
        return None, str(e)

paths = df_labels['image_path'].tolist()
print(f'Extracting features for {len(paths)} images in parallel...')
t0 = time.time()

results = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(safe_extract)(p) for p in paths
)

elapsed = time.time() - t0
print(f'\n✓ Extraction complete in {elapsed/60:.1f} min '
       f'({elapsed/len(paths)*1000:.0f} ms/image average)')

Extracting features for 6021 images in parallel...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    2.0s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    2.4s
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    3.5s
[Parallel(n_jobs=-1)]: Done  49 tasks      | elapsed:    4.2s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:    4.9s
[Parallel(n_jobs=-1)]: Done  81 tasks      | elapsed:    6.1s
[Parallel(n_jobs=-1)]: Done  98 tasks      | elapsed:    7.0s
[Parallel(n_jobs=-1)]: Done 117 tasks      | elapsed:    8.1s
[Parallel(n_jobs=-1)]: Done 136 tasks      | elapsed:    9.2s
[Parallel(n_jobs=-1)]: Done 157 tasks      | elapsed:   10.3s
[Parallel(n_jobs=-1)]: Done 178 tasks      | elapsed:   11.7s
[Parallel(n_jobs=-1)]: Done 201 tasks      | elapsed:   12.7s
[Parallel(n_jobs=-1)]: Done 224 tasks      | elapsed:   14.2s
[Parallel(n_jobs=-1)]: Done 249 tasks      | elapsed:   15.6s
[Parallel(n_jobs=-1)]: Done 274 tasks      | elapsed:  


✓ Extraction complete in 5.7 min (57 ms/image average)


[Parallel(n_jobs=-1)]: Done 6021 out of 6021 | elapsed:  5.7min finished


## 6. Assemble Feature Matrix

In [9]:
# Separate features and errors
features_list = []
errors = []
for i, (feat, err) in enumerate(results):
    if err is None:
        features_list.append(feat)
    else:
        features_list.append(None)
        errors.append((i, paths[i], err))

print(f'Successful extractions: {sum(1 for f in features_list if f is not None)}')
print(f'Failed extractions:     {len(errors)}')

if errors:
    print('\nFirst few errors:')
    for i, p, err in errors[:5]:
        print(f'  [{i}] {p.name}: {err[:100]}')

Successful extractions: 6021
Failed extractions:     0


In [10]:
# Build feature matrix
n_total = len(features_list)
n_features = len(features_list[0])  # first successful

# Replace None (failed) with NaN array
feature_matrix = np.zeros((n_total, n_features), dtype=np.float32)
for i, f in enumerate(features_list):
    if f is not None:
        feature_matrix[i] = f.astype(np.float32)
    else:
        feature_matrix[i] = np.nan

print(f'Feature matrix shape: {feature_matrix.shape}')
print(f'Memory: {feature_matrix.nbytes / 1e6:.1f} MB')

Feature matrix shape: (6021, 6216)
Memory: 149.7 MB


## 7. Save Features

In [11]:
# Build column names matching Stage 1's convention
feature_columns = (
    [f'hog_{i:04d}'   for i in range(n_hog)] +
    [f'color_{i:03d}' for i in range(n_color)] +
    [f'lbp_{i:03d}'   for i in range(n_lbp)] +
    [f'photo_{i:02d}' for i in range(n_photo)]
)
assert len(feature_columns) == n_features, \
    f'Column count mismatch: {len(feature_columns)} vs {n_features}'

# Combine into a DataFrame: identifiers + features
df_feat = pd.DataFrame(feature_matrix, columns=feature_columns)
df_feat.insert(0, 'sample_token', df_labels['sample_token'].values)

# Save
features_path = FEATURES_DIR / 'features_full.csv'
df_feat.to_csv(features_path, index=False)
print(f'Saved → {features_path}')
print(f'  Shape: {df_feat.shape}')
print(f'  File size: {features_path.stat().st_size / 1e6:.1f} MB')

Saved → data\processed\v1.0-trainval\features\features_full.csv
  Shape: (6021, 6217)
  File size: 415.9 MB


## 8. Save Feature Metadata

In [12]:
metadata = {
    'dataset_version': DATASET_VERSION,
    'n_keyframes':     int(n_total),
    'n_features':      int(n_features),
    'feature_groups': {
        'hog':         {'n_features': int(n_hog),   'columns': f'hog_0000 to hog_{n_hog-1:04d}'},
        'color':       {'n_features': int(n_color), 'columns': f'color_000 to color_{n_color-1:03d}'},
        'lbp':         {'n_features': int(n_lbp),   'columns': f'lbp_000 to lbp_{n_lbp-1:03d}'},
        'photometric': {'n_features': int(n_photo), 'columns': f'photo_00 to photo_{n_photo-1:02d}'},
    },
    'image_size':      IMG_SIZE,
    'hog_params':      HOG_PARAMS,
    'color_bins':      COLOR_BINS,
    'lbp_radius':      LBP_RADIUS,
    'lbp_n_points':    LBP_N_POINTS,
    'lbp_method':      LBP_METHOD,
    'failed_extractions': len(errors),
}

with open(FEATURES_DIR / 'feature_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Saved → {FEATURES_DIR / "feature_metadata.json"}')
print()
print(json.dumps(metadata, indent=2))

Saved → data\processed\v1.0-trainval\features\feature_metadata.json

{
  "dataset_version": "v1.0-trainval",
  "n_keyframes": 6021,
  "n_features": 6216,
  "feature_groups": {
    "hog": {
      "n_features": 6084,
      "columns": "hog_0000 to hog_6083"
    },
    "color": {
      "n_features": 96,
      "columns": "color_000 to color_095"
    },
    "lbp": {
      "n_features": 26,
      "columns": "lbp_000 to lbp_025"
    },
    "photometric": {
      "n_features": 10,
      "columns": "photo_00 to photo_09"
    }
  },
  "image_size": 224,
  "hog_params": {
    "orientations": 9,
    "pixels_per_cell": [
      16,
      16
    ],
    "cells_per_block": [
      2,
      2
    ],
    "block_norm": "L2-Hys",
    "transform_sqrt": true,
    "feature_vector": true
  },
  "color_bins": 32,
  "lbp_radius": 3,
  "lbp_n_points": 24,
  "lbp_method": "uniform",
  "failed_extractions": 0
}


## 9. Sanity Checks

In [13]:
print('=== FEATURE MATRIX SANITY CHECKS ===\n')

# 1. No NaN values (failed extractions)
n_nan_rows = np.isnan(feature_matrix).any(axis=1).sum()
print(f'1. Rows with any NaN:  {n_nan_rows} (should be 0)')

# 2. No constant features
feat_std = feature_matrix[~np.isnan(feature_matrix).any(axis=1)].std(axis=0)
n_constant = (feat_std == 0).sum()
print(f'2. Constant features:  {n_constant} (should be small or 0)')

# 3. Feature value ranges
print(f'\n3. Feature value ranges per group:')
slices = {
    'HOG':   slice(0, n_hog),
    'Color': slice(n_hog, n_hog + n_color),
    'LBP':   slice(n_hog + n_color, n_hog + n_color + n_lbp),
    'Photo': slice(n_hog + n_color + n_lbp, n_features),
}
for name, sl in slices.items():
    arr = feature_matrix[~np.isnan(feature_matrix).any(axis=1)][:, sl]
    print(f'  {name:6s}: min={arr.min():.3f}, max={arr.max():.3f}, '
          f'mean={arr.mean():.3f}, std={arr.std():.3f}')

=== FEATURE MATRIX SANITY CHECKS ===

1. Rows with any NaN:  0 (should be 0)
2. Constant features:  0 (should be small or 0)

3. Feature value ranges per group:
  HOG   : min=0.000, max=1.000, mean=0.140, std=0.090
  Color : min=0.000, max=0.153, mean=0.010, std=0.012
  LBP   : min=0.003, max=0.598, mean=0.038, std=0.076
  Photo : min=0.012, max=1.000, mean=0.290, std=0.184


## 10. Compare with v1.0-mini Feature Matrix

In [14]:
mini_features_paths = [
    Path('data/processed/v1.0-mini/features/features_full.csv'),
    Path('data/features/features_full.csv'),  # legacy
]
df_mini_feat = None
for p in mini_features_paths:
    if p.exists():
        df_mini_feat = pd.read_csv(p, nrows=5)  # just peek at columns
        print(f'Loaded v1.0-mini feature header: {p}')
        break

if df_mini_feat is not None:
    mini_cols = set(df_mini_feat.columns)
    subset_cols = set(df_feat.columns)

    common = mini_cols & subset_cols
    mini_only = mini_cols - subset_cols
    subset_only = subset_cols - mini_cols

    print(f'\nColumn comparison:')
    print(f'  v1.0-mini columns:          {len(mini_cols)}')
    print(f'  v1.0-trainval subset cols:  {len(subset_cols)}')
    print(f'  Common columns:             {len(common)}')
    print(f'  Mini-only columns:          {len(mini_only)}')
    print(f'  Subset-only columns:        {len(subset_only)}')

    if mini_only or subset_only:
        print('\n⚠️ Column mismatch between mini and subset features!')
        if mini_only:
            print(f'  Mini-only (first 5):   {list(mini_only)[:5]}')
        if subset_only:
            print(f'  Subset-only (first 5): {list(subset_only)[:5]}')
    else:
        print('\n✓ Feature columns match exactly — fair comparison guaranteed')
else:
    print('v1.0-mini features not found — column comparison skipped')

v1.0-mini features not found — column comparison skipped


## 11. Next Steps

In [15]:
print('=' * 70)
print('FEATURE EXTRACTION COMPLETE')
print('=' * 70)
print()
print(f'Features matrix:  {feature_matrix.shape}')
print(f'Total features:   {n_features}')
print(f'Saved to:         {features_path}')
print(f'Runtime:          {elapsed/60:.1f} min')
print()
print('Next notebook: 03_splits.ipynb')
print('  - Scene-aware 70/15/15 single split')
print('  - Stratified on time_of_day to ensure each split has day AND night')
print('  - Runtime: <1 min')

FEATURE EXTRACTION COMPLETE

Features matrix:  (6021, 6216)
Total features:   6216
Saved to:         data\processed\v1.0-trainval\features\features_full.csv
Runtime:          5.7 min

Next notebook: 03_splits.ipynb
  - Scene-aware 70/15/15 single split
  - Stratified on time_of_day to ensure each split has day AND night
  - Runtime: <1 min
